In [2]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from tqdm import tqdm

import utility_functions as uf

In [3]:
uf.PATH

'../data/'

In [4]:
df_article = pd.read_csv(uf.PATH+"Maxime_2010.csv", sep=";", decimal=",")
df_institution = pd.read_csv(uf.PATH+"Maxime_institutions.csv", sep=";", decimal=",")
df_topics = pd.read_csv(uf.PATH+"Maxime_topics.csv", sep=";", decimal=",")
df_stats = pd.read_csv(uf.PATH+"Maxime_stats.csv", sep=";", decimal=",")

In [5]:
parqToRead=uf.PATH+'authorsPapersTopicsYearInstitutions.parquet'

table = pq.read_table(parqToRead, filters=[('year','=','2010')])
df = table.to_pandas()

In [6]:
table = pq.read_table(parqToRead, columns=["year"])
years = table.column("year").unique().to_pylist()

In [7]:
years_array = np.sort(np.array(years))
years_array[years_array >= '1900']

array(['1900', '1901', '1902', '1903', '1904', '1905', '1906', '1907',
       '1908', '1909', '1910', '1911', '1912', '1913', '1914', '1915',
       '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1923',
       '1924', '1925', '1926', '1927', '1928', '1929', '1930', '1931',
       '1932', '1933', '1934', '1935', '1936', '1937', '1938', '1939',
       '1940', '1941', '1942', '1943', '1944', '1945', '1946', '1947',
       '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955',
       '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963',
       '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971',
       '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979',
       '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987',
       '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003',
       '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011',
      

In [8]:
cleared_institutions_list = df_institution.ID.drop_duplicates().to_list()

In [9]:
lost_institutions = []
df_lost_list = []
for year in tqdm(years_array[years_array >= '1900']):
    table = pq.read_table(parqToRead, filters=[('year','=',year)])
    df = table.to_pandas()

    mask = ~df.institution.drop_duplicates().isin(cleared_institutions_list)
    idx = df.institution.drop_duplicates().index[mask]

    if idx.empty:
        continue

    lost_tmp = [lost for lost in df.loc[idx, "institution"].to_list() if lost not in lost_institutions]
    lost_institutions += lost_tmp

    df_lost_tmp = (
        df
        [["year", "institution", "country", "coord", "inst_name"]]
        .query("institution == @lost_institutions")
        .drop_duplicates("institution")
    )
    if not df_lost_tmp.empty:
        df_lost_list.append(df_lost_tmp)

100%|██████████| 126/126 [25:09<00:00, 11.98s/it]


In [11]:
df_lost_institutions = pd.concat(df_lost_list)

In [14]:
df_lost_institutions

,year,institution,country,coord,inst_name
480024527,1901,I4210094106,IM,"[54.15383, -4.48212]",Manx National Heritage
134086952,1903,I4210094106,IM,"[54.15383, -4.48212]",Manx National Heritage
288366448,1904,I4210094106,IM,"[54.15383, -4.48212]",Manx National Heritage
30051663,1906,I4210094106,IM,"[54.15383, -4.48212]",Manx National Heritage
383156222,1908,I4210094106,IM,"[54.15383, -4.48212]",Manx National Heritage
...,...,...,...,...,...
165325539,2024,I2799559700,CW,"[12.3672, -69.11596]",Caribbean Research and Management of Biodivers...
242935391,2024,I2802914177,SX,"[18.034698, -63.08521]",American University of Integrative Sciences
436433803,2024,I4210101860,IM,"[54.173332, -4.505897]",Noble’s Hospital
494963461,2024,I4210120506,JE,"[49.183975, -2.110121]",Sarossa (Jersey)


In [17]:
df_lost_institutions.drop_duplicates("institution")

,year,institution,country,coord,inst_name
480024527,1901,I4210094106,IM,"[54.15383, -4.48212]",Manx National Heritage
26837565,1913,I2802914177,SX,"[18.034698, -63.08521]",American University of Integrative Sciences
26837567,1919,I4210115098,CW,"[12.157489, -68.96119]",University of Curaçao
364202929,1925,I4210086638,CW,"[12.109084, -68.94017]",St. Elisabeth Hospital
586703316,1953,I4210101860,IM,"[54.173332, -4.505897]",Noble’s Hospital
448019900,1959,I2799559700,CW,"[12.3672, -69.11596]",Caribbean Research and Management of Biodivers...
584178500,1964,I290569820,AX,"[60.103756, 19.928415]",Åland University of Applied Sciences
195249230,1965,I132121772,JE,"[49.23472, -2.09]",Durrell Wildlife Conservation Trust
288307782,1965,I57851904,None,"[37.25403, -76.49689]",Virginia Institute of Marine Science
469786064,1970,I78768387,XK,"[42.67272, 21.16688]",University of Prishtina


In [18]:
(
    df_article
    .groupby("ID", as_index="False")
    .count()
    .query("Institution > 1")
)

,Institution,Topic
ID,,
W1000038347,2,2
W1000054759,2,2
W100013949,3,3
W100014131,4,4
W1000272840,2,2
...,...,...
W999795231,2,2
W999797341,2,2
W999824637,2,2


In [23]:
article_id = "W100013949"
display(
    df_article[df_article["ID"] == article_id]
)
display(
    df[df.id == article_id]
)

,ID,Institution,Topic
867032,W100013949,I131868736,T10496
867033,W100013949,I158708052,T10496
867034,W100013949,I4210110554,T10496


In [71]:
print("N articles df_article:", len(df_article.ID.unique()))
print("N articles df:", len(df.id.unique()))
print("Lost (?) articles:", len(df.id.unique()) - len(df_article.ID.unique()))

N articles df_article: 2424128
N articles df: 2424205
Lost (?) articles: 77


In [93]:
df.columns

Index(['authors', 'id', 'year', 'institution', 'country', 'coord', 'inst_name',
       'topic'],
      dtype='object')

In [112]:
df_joined = (
    df[["id", "institution", "inst_name", "authors", "country", "coord"]]
    .merge(df_article, left_on=["id", "institution"], right_on=["ID", "Institution"], how="outer")
)

In [113]:
(
    df_joined
    .query("institution.isna()")
)

,id,institution,inst_name,authors,country,coord,ID,Institution,Topic


In [119]:
(
    df_joined
    .query("Institution.isna()")
    .merge(df_institution[["ID", "Country", "Country_cleaned"]], left_on="Institution", right_on="ID", how="left", suffixes=("", "_inst"))
    # .query("country != \"XK\"")
    [["institution", "inst_name", "coord", "country"]]
    .drop_duplicates(subset=["institution", "country"])
    # .assign(country_name=lambda df: [uf.id2name_country[c] for c in df["country"]])
)

,institution,inst_name,coord,country
0,I57851904,Virginia Institute of Marine Science,"[37.25403, -76.49689]",None
1,I78768387,University of Prishtina,"[42.67272, 21.16688]",XK
8,I4210087365,University Clinical Center of Kosovo,"[42.64312, 21.163242]",XK
10,I290569820,Åland University of Applied Sciences,"[60.103756, 19.928415]",AX
11,I4210086638,St. Elisabeth Hospital,"[12.109084, -68.94017]",CW
16,I132121772,Durrell Wildlife Conservation Trust,"[49.23472, -2.09]",JE
41,I4210094106,Manx National Heritage,"[54.15383, -4.48212]",IM
52,I4210086961,University for Business and Technology,"[42.558846, 21.134537]",XK
57,I4210115098,University of Curaçao,"[12.157489, -68.96119]",CW
97,I4210123547,Kosovo Telecom (Kosovo),"[42.655396, 21.157534]",XK


In [120]:
uf.get_country_info("SX")

,name,region,sub-region,code
201,Sint Maarten (Dutch part),Americas,Latin America and the Caribbean,SX


In [38]:
(
    df_article
    .merge(df_topics[["ID", "Sub_ID"]].set_index("ID"),
           left_on="Topic", right_index=True, how="left")
    .merge(df_institution[["ID", "Country_cleaned"]].set_index("ID"),
           left_on="Institution", right_index=True, how="left")
    .rename(columns={"Country_cleaned": "Country"})
    # .drop(["Institution", "Topic"], axis=1)
)

,ID,Institution,Topic,Sub_ID,Country
0,W2083709839,I4210117448,T10084,2736,GB
1,W2089978005,I1298853749,T10084,2736,NO
2,W2089978005,I1281400175,T10084,2736,NO
3,W2089978005,I1333353642,T10084,2736,NO
4,W2089978005,I78037679,T10084,2736,NO
...,...,...,...,...,...
3554600,W1485987360,I181391015,T12391,2204,BR
3554601,W1494198199,I27577105,T12391,2204,IE
3554602,W1494198199,I200332995,T12391,2204,DE
3554603,W2029193739,I25757504,T12391,2204,CN
